# 04 — Self-Supervised Pretraining (Fish images only)\n
\n
Pretrain a backbone using fish images only (unlabeled) and then reuse it for taxonomy fine-tuning.\n
\n
Method: SimCLR-style contrastive learning (pure PyTorch) for device flexibility.

In [1]:
from pathlib import Path
import sys
import torch

ROOT = Path('..').resolve()
sys.path.append(str((ROOT / 'src').resolve()))

MANIFEST_DIR = (ROOT / 'data' / 'manifests').resolve()
OUT_DIR = (ROOT / 'outputs' / 'ssl_simclr').resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device(
    'cuda'
    if torch.cuda.is_available()
    else 'mps'
    if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()
    else 'cpu'
)
print('device:', device)
print('OUT_DIR:', OUT_DIR)


device: mps
OUT_DIR: /Users/sushantshah/Desktop/Fish Codes/outputs/ssl_simclr


In [2]:
from fish_ai.data.ssl_dataset import FishSSLManifestDataset
from fish_ai.models.simclr import SimCLRConfig
from fish_ai.train.ssl_simclr_train import TrainConfig, fit

train_manifest = MANIFEST_DIR / 'fishnet_taxonomy_train.jsonl'
print('train_manifest exists:', train_manifest.exists())

ds_ssl = FishSSLManifestDataset(train_manifest, image_size=224)
len(ds_ssl)


train_manifest exists: True


37700

In [3]:
ssl_cfg = SimCLRConfig(backbone='resnet50', proj_dim=128, proj_hidden_dim=2048, temperature=0.2)
cfg = TrainConfig(num_epochs=5, batch_size=64, num_workers=2, lr=3e-4)

model = fit(ds_ssl, ssl_cfg=ssl_cfg, cfg=cfg, device=device)

ckpt_path = OUT_DIR / 'simclr_resnet50.pt'
torch.save({'model_state': model.state_dict(), 'ssl_cfg': ssl_cfg.__dict__, 'cfg': cfg.__dict__}, ckpt_path)
print('saved:', ckpt_path)


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /Users/sushantshah/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


URLError: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1000)>